# 🔍 Swiss Legal Retrieval — LanceDB Index

Builds a hybrid search index (dense vectors + BM25) over the Swiss legal
corpus. Output: a `lancedb_data/` directory with a table that supports
vector search, full-text search, and hybrid fusion in a single query.

**This is a utility notebook** — attach it to your submission notebook via
`Add Input`. All the expensive indexing work happens once, here.

## 🎯 What it does
1. 📦 Installs lancedb + tantivy from the wheels utility notebook
2. 📥 Streams `corpus.parquet` (6.4 GB, 2.9M rows) into LanceDB
3. 🗂️ Creates BM25 full-text search index on text + title
4. ✅ Verifies both vector and FTS search work

## 💡 Why LanceDB
- **fp16 native** — vectors stored and searched in fp16, no fp32 cast
- **Built-in BM25** — Tantivy under the hood, no external lib
- **Hybrid search in one call** — `query_type="hybrid"` fuses vector + FTS
- **Pure Python, local files** — no server binary, no staging hacks
- **Fits Kaggle's 19.5 GB limit** — ~8-10 GB total output

## 📜 The road to LanceDB

Before landing here, I tried Weaviate. Twice. First attempt OOMed during
ingest because fp32 vectors plus HNSW overhead blew past 30 GB RAM. Second
attempt finished ingest but Weaviate requires fp32 input, so the index
needed 20+ GB of disk — over the Kaggle output limit. The full-text index
was corrupt by the end because shards went read-only under disk pressure.

LanceDB sidesteps both problems: fp16 input cuts disk in half, and the
index builds after ingest instead of during it, so there's no
compounding HNSW slowdown during insertion.

## 👋 About me

Publishing this as a community hobby project — I'm **not eligible for the
prize pool**. Feel free to fork, modify, learn from it.

**I'm an Agentic AI Developer.** I work across the whole modern LLM
application stack.

**I'm looking for work** 🔍. If you're hiring for Agentic AI / LLM
application engineering roles, please reach out:

📧 **re.azami@gmail.com**

Remote, hybrid, or on-site — all welcome.

## 📦 Install packages

Install lancedb + tantivy from the wheels notebook. Also locate the
corpus parquet from the corpus notebook.

In [1]:
import sys, subprocess, os
from pathlib import Path

WHEELS_UTIL = Path("/kaggle/usr/lib/notebooks/rezaazami/utility_swiss_legal_wheels_and_models")
CORPUS_UTIL = Path("/kaggle/usr/lib/notebooks/rezaazami/utility_swiss_legal_corpus")

assert WHEELS_UTIL.exists(), f"Wheels notebook not attached at {WHEELS_UTIL}"
assert CORPUS_UTIL.exists(), f"Corpus notebook not attached at {CORPUS_UTIL}"

WHEELS = WHEELS_UTIL / "wheels"
PARQUET = CORPUS_UTIL / "corpus.parquet"

assert WHEELS.exists(), f"Missing: {WHEELS}"
assert PARQUET.exists(), f"Missing: {PARQUET}"

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "--no-index", "--find-links", str(WHEELS),
    "lancedb", "tantivy",
])

import lancedb, tantivy
print(f"lancedb: {lancedb.__version__}")
print(f"tantivy available")
print(f"corpus.parquet: {os.path.getsize(PARQUET)/1e9:.2f} GB")

lancedb: 0.30.2
tantivy available
corpus.parquet: 6.45 GB


## 🧪 Sanity probe

Before pumping 2.9M rows in, verify LanceDB actually accepts fp16 vectors
and round-trips them. This catches API issues before wasting an hour on
ingest.

In [2]:
import lancedb
import pyarrow as pa
import numpy as np
import shutil
from pathlib import Path

PROBE = Path("/tmp/lance_probe")
if PROBE.exists():
    shutil.rmtree(PROBE)

db_probe = lancedb.connect(str(PROBE))

# Explicit schema — this is the key. Without it, LanceDB defaults to float32.
probe_schema = pa.schema([
    pa.field("text",   pa.string()),
    pa.field("vector", pa.list_(pa.float16(), 1024)),
])

test_data = [
    {"text": f"test {i}", "vector": np.random.randn(1024).astype(np.float16)}
    for i in range(100)
]
tbl_probe = db_probe.create_table(
    "probe",
    test_data,
    schema=probe_schema,     # ← explicit schema forces fp16
    mode="overwrite",
)

q = np.random.randn(1024).astype(np.float16)
res = tbl_probe.search(q).limit(3).to_pandas()

result_dtype = np.asarray(res.iloc[0]["vector"]).dtype
print(f"Rows inserted: {tbl_probe.count_rows()}")
print(f"Search works: {len(res)} results")
print(f"Vector dtype in result: {result_dtype}")

# Also verify the table schema itself
print(f"\nTable schema:")
print(tbl_probe.schema)

assert result_dtype == np.float16, f"fp16 not preserved! got {result_dtype}"

del db_probe, tbl_probe
shutil.rmtree(PROBE)
print("\n✓ Probe passed — LanceDB handles fp16 correctly")

Rows inserted: 100
Search works: 3 results
Vector dtype in result: float16

Table schema:
text: string
vector: fixed_size_list<item: halffloat>[1024]
  child 0, item: halffloat

✓ Probe passed — LanceDB handles fp16 correctly


[2026-04-17T04:50:13Z WARN  lance::dataset::write::insert] No existing dataset at /tmp/lance_probe/probe.lance, it will be created


## 📥 Streaming ingest

Read `corpus.parquet` in row groups (never full dataset in RAM), convert
each batch to LanceDB records, append to the table.

- **First row group** creates the table with `mode="overwrite"`
- **Subsequent row groups** append via `table.add()`
- Vectors stay fp16 throughout — no conversion

In [3]:
import lancedb
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import time, gc, shutil
from pathlib import Path

LANCE_DIR = Path("/kaggle/working/lancedb_data")
if LANCE_DIR.exists():
    shutil.rmtree(LANCE_DIR)

db = lancedb.connect(str(LANCE_DIR))

# Explicit schema — forces fp16 for vectors, matches the corpus.parquet schema
table_schema = pa.schema([
    pa.field("citation",    pa.string()),
    pa.field("text",        pa.string()),
    pa.field("title",       pa.string()),
    pa.field("source_type", pa.string()),
    pa.field("chunk_id",    pa.int32()),
    pa.field("vector",      pa.list_(pa.float16(), 1024)),
])

pf = pq.ParquetFile(PARQUET)
total_rows = pf.metadata.num_rows
n_groups = pf.num_row_groups
print(f"Parquet: {total_rows:,} rows in {n_groups} row groups")

t0 = time.time()
table = None
processed = 0

for rg_idx in range(n_groups):
    rg = pf.read_row_group(rg_idx).to_pandas()
    
    records = []
    for _, row in rg.iterrows():
        vec = np.asarray(row["vector"])
        if vec.dtype != np.float16:
            vec = vec.astype(np.float16)
        records.append({
            "citation":    str(row["citation"]),
            "text":        str(row["text"]),
            "title":       str(row["title"]) if row["title"] is not None else "",
            "source_type": str(row["source_type"]),
            "chunk_id":    int(row["chunk_id"]),
            "vector":      vec,
        })
    
    if table is None:
        table = db.create_table(
            "swiss_legal",
            records,
            schema=table_schema,    # ← explicit schema
            mode="overwrite",
        )
    else:
        table.add(records)
    
    processed += len(rg)
    elapsed = time.time() - t0
    rate = processed / elapsed
    eta_min = (total_rows - processed) / rate / 60
    print(f"  [{rg_idx+1:>2}/{n_groups}] {processed:>10,} / {total_rows:,} | "
          f"{rate:>6.0f} rows/s | ETA: {eta_min:>5.1f} min")
    
    del rg, records
    gc.collect()

elapsed = time.time() - t0
print(f"\nIngest complete: {processed:,} rows in {elapsed/60:.1f} min")
print(f"Table row count: {table.count_rows():,}")
print(f"\nTable schema:")
print(table.schema)

Parquet: 2,929,142 rows in 30 row groups


[2026-04-17T04:50:31Z WARN  lance::dataset::write::insert] No existing dataset at /kaggle/working/lancedb_data/swiss_legal.lance, it will be created


  [ 1/30]    100,000 / 2,929,142 |   5457 rows/s | ETA:   8.6 min
  [ 2/30]    200,000 / 2,929,142 |   5780 rows/s | ETA:   7.9 min
  [ 3/30]    300,000 / 2,929,142 |   5824 rows/s | ETA:   7.5 min
  [ 4/30]    400,000 / 2,929,142 |   5829 rows/s | ETA:   7.2 min
  [ 5/30]    500,000 / 2,929,142 |   5868 rows/s | ETA:   6.9 min
  [ 6/30]    600,000 / 2,929,142 |   5926 rows/s | ETA:   6.6 min
  [ 7/30]    700,000 / 2,929,142 |   5939 rows/s | ETA:   6.3 min
  [ 8/30]    800,000 / 2,929,142 |   5960 rows/s | ETA:   6.0 min
  [ 9/30]    900,000 / 2,929,142 |   5991 rows/s | ETA:   5.6 min
  [10/30]  1,000,000 / 2,929,142 |   6023 rows/s | ETA:   5.3 min
  [11/30]  1,100,000 / 2,929,142 |   6032 rows/s | ETA:   5.1 min
  [12/30]  1,200,000 / 2,929,142 |   5941 rows/s | ETA:   4.9 min
  [13/30]  1,300,000 / 2,929,142 |   5780 rows/s | ETA:   4.7 min
  [14/30]  1,400,000 / 2,929,142 |   5780 rows/s | ETA:   4.4 min
  [15/30]  1,500,000 / 2,929,142 |   5789 rows/s | ETA:   4.1 min
  [16/30] 

## 🗂️ Build the vector index (IVF_HNSW_SQ)

Without this, vector search does a brute-force scan over all 2.9M rows
(~1-3 seconds per query). With IVF_HNSW_SQ, queries drop to ~50-150ms
while keeping ~98-99% recall.

### What this does

**IVF** partitions vectors into 256 clusters. At query time, only the
nprobes closest partitions are searched — narrowing 2.9M → ~230k candidates.

**HNSW** (inside each partition) builds a graph for fast traversal within
the partition. Much faster than flat scan per partition.

**SQ** (Scalar Quantization) compresses each dimension from fp16 → int8,
halving the index size with near-zero recall loss.

### Parameters

- `num_partitions=256` — ~√(2.9M / 45) rounded, following LanceDB's
  rule-of-thumb for ~11k vectors per partition
- `m=20` — HNSW graph connectivity per node
- `ef_construction=300` — build-time effort (higher = better graph quality)

### Expected build time

Typically 15-30 minutes for 2.9M × 1024-dim vectors on Kaggle CPU.
IVF training is the slow part (k-means on a sample); HNSW graph build
is relatively fast since each partition is small.

In [4]:
import time

print("Building IVF_HNSW_SQ index...")
print(f"  num_partitions: 256")
print(f"  m: 20")
print(f"  ef_construction: 300")
print(f"  metric: cosine\n")

t0 = time.time()
table.create_index(
    vector_column_name="vector",
    metric="cosine",
    index_type="IVF_HNSW_SQ",
    num_partitions=256,
    m=20,
    ef_construction=300,
    replace=True,
)
elapsed = time.time() - t0
print(f"Vector index built in {elapsed/60:.1f} min ({elapsed:.0f}s)")

# Show index stats
try:
    stats = table.index_stats("vector_idx")
    print(f"\nIndex stats:")
    print(f"  Indexed rows: {stats.num_indexed_rows:,}")
    print(f"  Unindexed rows: {stats.num_unindexed_rows:,}")
    print(f"  Index type: {stats.index_type}")
    print(f"  Distance type: {stats.distance_type}")
except Exception as e:
    print(f"(Could not fetch index stats: {e})")

Building IVF_HNSW_SQ index...
  num_partitions: 256
  m: 20
  ef_construction: 300
  metric: cosine

Vector index built in 35.6 min (2138s)

Index stats:
  Indexed rows: 2,929,142
  Unindexed rows: 0
  Index type: IVF_HNSW_SQ
  Distance type: cosine


## 🗂️ Build the BM25 index

Create the full-text search index on `text` and `title`. `replace=True`
wipes any existing index and rebuilds fresh. This is where Tantivy tokenizes
every document and builds the inverted index.

Expect 5-15 minutes for ~3M documents.

In [5]:
import time

t0 = time.time()

# Index on text (main content)
table.create_fts_index("text", replace=True)
print(f"FTS index on 'text' built in {(time.time()-t0)/60:.1f} min")

t1 = time.time()
# Index on title (short, for laws only)
table.create_fts_index("title", replace=True)
print(f"FTS index on 'title' built in {(time.time()-t1)/60:.1f} min")

print(f"\nTotal FTS build time: {(time.time()-t0)/60:.1f} min")

FTS index on 'text' built in 3.7 min
FTS index on 'title' built in 0.0 min

Total FTS build time: 3.8 min


## ✅ Verify search works

Quick end-to-end test: vector search, FTS search, and hybrid search with
a realistic German legal query.

In [6]:
import numpy as np

# Vector search — random vector, just checks the plumbing
test_vec = np.random.randn(1024).astype(np.float16)
vec_results = table.search(test_vec).limit(3).to_pandas()
print("Vector search test:")
print(f"  Results: {len(vec_results)}")
print(f"  Columns: {list(vec_results.columns)}")
print(f"  First citation: {vec_results.iloc[0]['citation']}")

# FTS search — a real German legal term
fts_results = table.search("Bundesgericht", query_type="fts").limit(3).to_pandas()
print(f"\nFTS search test ('Bundesgericht'):")
print(f"  Results: {len(fts_results)}")
if len(fts_results) > 0:
    print(f"  First citation: {fts_results.iloc[0]['citation']}")
    print(f"  First text (first 100 chars): {fts_results.iloc[0]['text'][:100]}")

# Hybrid search — both at once
hybrid_results = (
    table.search(query_type="hybrid")
    .text("Bundesgericht")
    .vector(test_vec)
    .limit(3)
    .to_pandas()
)
print(f"\nHybrid search test:")
print(f"  Results: {len(hybrid_results)}")
if len(hybrid_results) > 0:
    print(f"  First citation: {hybrid_results.iloc[0]['citation']}")

Vector search test:
  Results: 3
  Columns: ['citation', 'text', 'title', 'source_type', 'chunk_id', 'vector', '_distance']
  First citation: Art. 138 Abs. 1 817.022.12

FTS search test ('Bundesgericht'):
  Results: 3
  First citation: Art. 80f Abs. 2 PVBger
  First text (first 100 chars): 2 Die Adressaten des Bulletins des Generalsekretariats sind:a. die Mitglieder und die Angestellten d

Hybrid search test:
  Results: 3
  First citation: Art. 138 Abs. 1 817.022.12


## 📦 Verify output size

Confirm the total output fits under Kaggle's 19.5 GB limit.

In [7]:
import numpy as np
import time

test_vec = np.random.randn(1024).astype(np.float16)

# Vector search — default settings
t0 = time.time()
vec_default = table.search(test_vec).limit(10).to_pandas()
print(f"Default (IVF_HNSW_SQ): {len(vec_default)} results in {(time.time()-t0)*1000:.0f}ms")

# Vector search — higher recall settings
t0 = time.time()
vec_tuned = (
    table.search(test_vec)
    .nprobes(40)
    .ef(100)
    .refine_factor(2)
    .limit(10)
    .to_pandas()
)
print(f"Tuned (nprobes=40, ef=100, refine=2): {len(vec_tuned)} results in {(time.time()-t0)*1000:.0f}ms")

# Brute-force bypass — should be slower but 100% accurate
t0 = time.time()
vec_exact = table.search(test_vec).bypass_vector_index().limit(10).to_pandas()
print(f"Brute-force (bypass): {len(vec_exact)} results in {(time.time()-t0)*1000:.0f}ms")

# FTS search
t0 = time.time()
fts_results = table.search("Bundesgericht", query_type="fts").limit(10).to_pandas()
print(f"\nFTS search ('Bundesgericht'): {len(fts_results)} results in {(time.time()-t0)*1000:.0f}ms")

# Hybrid
t0 = time.time()
hybrid_results = (
    table.search(query_type="hybrid")
    .text("Bundesgericht")
    .vector(test_vec)
    .limit(10)
    .to_pandas()
)
print(f"Hybrid search: {len(hybrid_results)} results in {(time.time()-t0)*1000:.0f}ms")

Default (IVF_HNSW_SQ): 10 results in 256ms
Tuned (nprobes=40, ef=100, refine=2): 10 results in 566ms
Brute-force (bypass): 10 results in 24525ms

FTS search ('Bundesgericht'): 10 results in 64ms
Hybrid search: 10 results in 67ms
